# Word Tagger
In a tagged corpus every word carries its part of speech: `power_NN1` is a singular noun, `taxing_VVG` a verb ending in -ing. This notebook reads a real tagged corpus, `CLAWS5inTASA.dat`, with **one SNOBOL4 pattern**. An action inside the pattern, `token()`, files each word under its sentence number and its tag at the moment the pattern matches it. Run the cells in order.

## 1. The pattern, and the run
The pattern reads a number or a word with its tag, over and over, to the end of the file. The last lines open `CLAWS5inTASA.dat`, beside this notebook, and match the whole of it.

In [ ]:
        DEFINE('token()')  :(token_end)
token   num = +num
        mem = IDENT(mem) TABLE()
        mem[num] = IDENT(mem[num]) TABLE()
        mem[num][wrd] = IDENT(mem[num][wrd]) TABLE()
        mem[num][wrd][tag] = IDENT(mem[num][wrd][tag]) 0
        mem[num][wrd][tag] = mem[num][wrd][tag] + 1
        token = .dummy  :(NRETURN)
token_end
        &TRIM   =   0
        claws   =   POS(0)
+                   ARBNO(
+                     FENCE(
+                       (SPAN('0123456789') . num) '_CRD :_PUN'
+                     | (NOTANY('_') BREAK('_')) . wrd
+                       '_'
+                       (ANY(&UCASE) SPAN('0123456789' &UCASE)) . tag . *token()
+                     )
+                     SPAN(' ' CHAR(10))
+                   )
+                   RPOS(0)
        INPUT(.INPUT, 9, 'CLAWS5inTASA.dat [-r200000]')
        src = INPUT                                  :F(nomatch)
        src ? claws                                  :F(nomatch)
        OUTPUT = 'matched bytes=' SIZE(src)          :(shown)
nomatch OUTPUT = 'Pattern match failed'
shown

## 2. What it found
The words are now in a table of tables, `mem`, by sentence, word and tag. Here is the first sentence, its words in alphabetical order.

In [ ]:
        w = SORT(mem[1])
        PROTOTYPE(w) ? BREAK(',') . n
        OUTPUT = n ' different words in the first sentence; the first few, with their tags:'
        i = 0
loop    i = LT(i, 6) i + 1                           :F(done)
        t = SORT(w[i,2])
        OUTPUT = w[i,1] ' is tagged ' t[1,1]         :(loop)
done    OUTPUT = num ' sentences in all'